# CEG-WM prospective Content texture stratification v1
User-only exploratory analysis. This notebook does not change any method, Gate, or scientific status. Run once; do not retry or resume.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except BaseException:
    print('CEGWM_TEXTURE_HANDOFF_FAILURE {"error_class":"OtherOperationalError","execution_exact":"4bfcb3f80dedd83b855c221b67e23b5d5cc14210","run_id":"content-texture-stratification-v1-3bf6552daa78-805bc21e173a","stage":"drive_mount","status":"operational_failure"}', flush=True)
    HANDOFF_FAILED = True
else:
    HANDOFF_FAILED = False

import json
import os
from pathlib import Path
import re
import subprocess
import sys

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
TARGET_BRANCH = 'stage-a-content-texture-stratification-v1'
EXPECTED_EXACT = '4bfcb3f80dedd83b855c221b67e23b5d5cc14210'
RUNNER_MODULE = 'experiments.run_content_texture_stratification_v1'
RESULT_PREFIX = 'CEGWM_TEXTURE_RESULT'
FAILURE_PREFIX = 'CEGWM_TEXTURE_HANDOFF_FAILURE'
ARTIFACT_PREFIX = 'CEGWM_TEXTURE_ARTIFACT'
ANALYSIS_ID = 'content_texture_stratification_v1'
CLAIM_CEILING = 'exploratory_prospective_texture_stratification_only'
PROTOCOL_DIGEST = '3bf6552daa78ea11b3038d682f2ec623d011f4cbb5709233b1702fae1437a70e'
PUBLIC_KEY_DIGEST = '805bc21e173a83898f3b7034d75e6ed02f65894a6885377d9659ee3091b4dd77'
RUN_ID = 'content-texture-stratification-v1-3bf6552daa78-805bc21e173a'
SOURCE = Path('/content/cegwm-content-texture-stratification-v1-source')
LOCAL = Path('/content/cegwm-content-texture-stratification-v1-local')
SINK = Path('/content/drive/MyDrive/CEG-WM/content_texture_stratification_v1')
PROVENANCE = Path('/content/drive/MyDrive/CEG-WM')
RUN_ROOT = SINK / EXPECTED_EXACT / RUN_ID
TERMINAL = RUN_ROOT / 'terminal'
TERMINAL_ZIP = TERMINAL / (RUN_ID + '.zip')
TERMINAL_SHA = TERMINAL / (RUN_ID + '.zip.sha256')
RUNNER_ATTEMPTED = False
ACCEPTED_ARTIFACT = None
_ALLOWED_ERRORS = {'CalledProcessError', 'FileExistsError', 'ImportError', 'ModuleNotFoundError', 'OSError', 'RuntimeError', 'TypeError', 'UnicodeDecodeError', 'ValueError'}

def fail(stage, error_class='RuntimeError'):
    global HANDOFF_FAILED, ACCEPTED_ARTIFACT
    ACCEPTED_ARTIFACT = None
    if HANDOFF_FAILED:
        return
    if error_class not in _ALLOWED_ERRORS:
        error_class = 'OtherOperationalError'
    HANDOFF_FAILED = True
    payload = {'status': 'operational_failure', 'analysis_id': ANALYSIS_ID, 'execution_exact': EXPECTED_EXACT, 'run_id': RUN_ID, 'stage': stage, 'error_class': error_class}
    line = FAILURE_PREFIX + ' ' + json.dumps(payload, sort_keys=True, separators=(',', ':'))
    if len(line.encode('utf-8')) <= 4096:
        print(line, flush=True)

def git(*args):
    return subprocess.run(['git', *args], cwd=SOURCE, check=True, capture_output=True, text=True).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if SOURCE.exists() or LOCAL.exists() or RUN_ROOT.exists():
            raise FileExistsError
    except BaseException as error:
        fail('initial_only_path_validation', type(error).__name__)


In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(['git', 'clone', '--no-single-branch', '--branch', TARGET_BRANCH, REPO_URL, str(SOURCE)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if git('branch', '--show-current') != TARGET_BRANCH or git('rev-parse', 'HEAD') != EXPECTED_EXACT or git('status', '--porcelain') != '' or LOCAL.exists() or RUN_ROOT.exists():
            raise RuntimeError
        subprocess.run([sys.executable, '-m', 'pip', 'install', str(SOURCE)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if git('branch', '--show-current') != TARGET_BRANCH or git('rev-parse', 'HEAD') != EXPECTED_EXACT or git('status', '--porcelain') != '' or LOCAL.exists() or RUN_ROOT.exists():
            raise RuntimeError
    except BaseException as error:
        fail('checkout_install_identity_validation', type(error).__name__)


In [ ]:
from google.colab import userdata

RESULT_FIELDS = {'status', 'claim_ceiling', 'exact', 'protocol_digest', 'run_id', 'terminal_sha256'}
CAPTURE_LIMIT = 4096

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ''
    hf_token = ''
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_error = None
    try:
        if git('branch', '--show-current') != TARGET_BRANCH or git('rev-parse', 'HEAD') != EXPECTED_EXACT or git('status', '--porcelain') != '' or LOCAL.exists() or RUN_ROOT.exists():
            raise RuntimeError
        root_key = userdata.get('CEG_WM_ROOT_KEY')
        hf_token = userdata.get('HF_TOKEN')
        if not isinstance(root_key, str) or not root_key.strip() or not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        secret_markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
        runner_env = {name: value for name, value in os.environ.items() if not any(marker in name.upper() for marker in secret_markers)}
        runner_env['CEG_WM_ROOT_KEY'] = root_key
        runner_env['HF_TOKEN'] = hf_token
        root_key = ''
        hf_token = ''
        command = [sys.executable, '-m', RUNNER_MODULE, '--repo-root', str(SOURCE), '--expected-exact', EXPECTED_EXACT, '--local-work-root', str(LOCAL), '--artifact-sink', str(SINK), '--provenance-root', str(PROVENANCE)]
        process = subprocess.Popen(command, cwd=SOURCE, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        runner_env.pop('CEG_WM_ROOT_KEY', None)
        runner_env.pop('HF_TOKEN', None)
        runner_env = None
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
                if process.poll() is None:
                    process.kill()
        runner_rc = process.wait()
    except BaseException as error:
        launch_error = type(error).__name__
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ''
        hf_token = ''
        if runner_env is not None:
            runner_env.pop('CEG_WM_ROOT_KEY', None)
            runner_env.pop('HF_TOKEN', None)
        runner_env = None

    try:
        if launch_error is not None:
            raise RuntimeError
        if capture_overflow:
            raise RuntimeError
        if runner_rc not in (0, 2):
            raise RuntimeError
        if len(captured) > CAPTURE_LIMIT or captured.count(b'\n') != 1 or not captured.endswith(b'\n'):
            raise RuntimeError
        result_line = captured[:-1].decode('utf-8', errors='strict')
        if not result_line.startswith(RESULT_PREFIX + ' '):
            raise RuntimeError
        result = json.loads(result_line.split(' ', 1)[1])
        if not isinstance(result, dict) or set(result) != RESULT_FIELDS or any(not isinstance(result[name], str) for name in RESULT_FIELDS):
            raise RuntimeError
        if (runner_rc == 0 and result['status'] != 'analysis_complete') or (runner_rc == 2 and result['status'] != 'not_interpretable'):
            raise RuntimeError
        if result['claim_ceiling'] != CLAIM_CEILING or result['exact'] != EXPECTED_EXACT or result['protocol_digest'] != PROTOCOL_DIGEST or result['run_id'] != RUN_ID or re.fullmatch(r'[0-9a-f]{64}', result['terminal_sha256']) is None:
            raise RuntimeError
        if not TERMINAL_ZIP.is_file() or not TERMINAL_SHA.is_file() or TERMINAL_SHA.stat().st_size > CAPTURE_LIMIT:
            raise RuntimeError
        binding = TERMINAL_SHA.read_text(encoding='ascii')
        if binding != result['terminal_sha256'] + '  ' + TERMINAL_ZIP.name + '\n':
            raise RuntimeError
        ACCEPTED_ARTIFACT = {'status': result['status'], 'claim_ceiling': result['claim_ceiling'], 'execution_exact': result['exact'], 'protocol_digest': result['protocol_digest'], 'run_id': result['run_id'], 'terminal_sha256': result['terminal_sha256'], 'archive_path': str(TERMINAL_ZIP), 'sidecar_path': str(TERMINAL_SHA)}
    except BaseException:
        ACCEPTED_ARTIFACT = None
        if launch_error is not None:
            fail('runner_launch', launch_error)
        elif capture_overflow:
            fail('runner_stdout_overflow')
        elif runner_rc not in (0, 2):
            fail('runner_return_code')
        else:
            fail('runner_result_or_artifact_validation')
    finally:
        captured.clear()


In [ ]:
artifact = globals().get('ACCEPTED_ARTIFACT')
if not HANDOFF_FAILED and isinstance(artifact, dict):
    try:
        expected_fields = {'status', 'claim_ceiling', 'execution_exact', 'protocol_digest', 'run_id', 'terminal_sha256', 'archive_path', 'sidecar_path'}
        if set(artifact) != expected_fields or artifact['archive_path'] != str(TERMINAL_ZIP) or artifact['sidecar_path'] != str(TERMINAL_SHA):
            raise RuntimeError
        if not TERMINAL_ZIP.is_file() or not TERMINAL_SHA.is_file() or TERMINAL_SHA.stat().st_size > 4096:
            raise RuntimeError
        if TERMINAL_SHA.read_text(encoding='ascii') != artifact['terminal_sha256'] + '  ' + TERMINAL_ZIP.name + '\n':
            raise RuntimeError
        receipt = {'status': 'artifact_pair_saved', 'analysis_status': artifact['status'], 'claim_ceiling': artifact['claim_ceiling'], 'execution_exact': artifact['execution_exact'], 'protocol_digest': artifact['protocol_digest'], 'run_id': artifact['run_id'], 'terminal_sha256': artifact['terminal_sha256'], 'archive_path': artifact['archive_path'], 'sidecar_path': artifact['sidecar_path']}
        line = ARTIFACT_PREFIX + ' ' + json.dumps(receipt, sort_keys=True, separators=(',', ':'))
        if len(line.encode('utf-8')) > 4096:
            raise RuntimeError
        print(line, flush=True)
    except BaseException as error:
        ACCEPTED_ARTIFACT = None
        fail('artifact_pair_validation', type(error).__name__)
